# Proposed generator: training and generation

Trains the hidden Markov model (HMM) and the mixture density network (MDN) on the training flows of one dataset and split, generates one synthetic flow for every test flow, and saves the trained model. `scripts/run_matrix.py` runs this notebook with papermill and overrides the parameters below; the defaults are for a run by hand.

Outputs in `OUT_DIR`: `<MODEL_KEY>.csv` (synthetic flows), `test.csv` (real test flows), `split.json` (train and test flow ids) and `artifacts/<MODEL_KEY>/` (HMM, scalers, MDN weights and `meta.json`).

In [ ]:
SEED                 = 0
DATASET              = 'facebook_video'
RAW_PATH_STR         = 'data/facebook_video.csv'
OUT_DIR              = 'runs/facebook_video/seed0'
MODEL_KEY            = 'proposed'
TEST_SIZE            = 0.3          # share of the flows in the test set
MIN_TRAIN_PKT_SHARE  = 0.6          # the split is redrawn until the training flows hold this share of the packets
VAL_FRACTION         = 0.0          # > 0: this share of the training flows is held out for the validation NLL
ABLATION_NO_IDLE     = False        # True: no idle state (variant A)
ABLATION_NO_FLOW_LEN = False        # True: no flow-length input (variant B)
DIST_FAMILY          = 'student'    # 'student', or 'gaussian' (variant C)
K_CORE               = 4            # core HMM states; the idle state is added on top
N_MIX_MDN            = 16           # M, mixture components of the MDN
HIDDEN_MDN           = 128          # H, width of the two hidden layers
TAIL_Q               = 0.998        # quantile of log(1 + IAT) of the training packets that defines tau
TEMPERATURE_IDLE     = 1.0          # optional temperature of the mixture weights in the idle state (1.0: none)
FLOW_LEN_MODE        = 'oracle'     # lengths of the generated flows: 'oracle' (test flows) or 'empirical'
EPOCHS_MDN           = 90           # training epochs of the MDN

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import joblib
import numpy as np
import torch
import torch.nn as nn
from hmmlearn import hmm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()   # run from the repository root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'scripts'))
from data_utils import atomic_write_csv, create_or_load_split, load_packets
from hmm_mdn import (GlobalStateLenMDN, PosteriorDataset, flow_length_selector, generate_flows,
                     nll_gaussian, nll_student_t)
from safe_hmm import SafeGMMHMM
from split_rule import constrained_split

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'     # deterministic cuBLAS kernels
torch.use_deterministic_algorithms(True)
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# HMM
N_MIX_HMM     = 1                       # Gaussian components per state
MAX_ITER_HMM  = 200                     # maximum number of EM iterations
MIN_COVAR     = 4e-3                    # variance floor of the emissions
INIT_VAR_EPS  = 1e-4                    # added to the k-means cluster variances at initialization
IDLE_INIT_VAR = 2.0                     # initial variance of the idle state in both coordinates
LAMBDA_SELF, LAMBDA_OFF  = 5.0, 0.5     # Dirichlet pseudo-counts of the transition matrix: core rows
LAMBDA_IDLE, LAMBDA_LEAK = 25.0, 0.1    # idle row
HMM_RESTARTS  = 5                       # further initializations if EM degenerates
# MDN
MDN_BATCHSIZE = 4096
LR_MDN        = 1e-3
GRAD_CLIP     = 5.0
THRESH_GAMMA_CORE = 1e-4                # minimum posterior of a packet in the MDN data of a core state
THRESH_GAMMA_IDLE = 1e-6                # the same for the idle state
if DIST_FAMILY not in ('student', 'gaussian'):
    raise ValueError("DIST_FAMILY must be 'student' or 'gaussian'")

In [ ]:
# 1  Packets: cleaning, log transforms and the flow-length feature
df = load_packets(RAW_PATH_STR)
df['payload_log']   = np.log1p(df['payload_length'])
df['time_diff_log'] = np.log1p(df['time_diff'])
df['flow_len_log']  = np.log1p(df.groupby('flow_id')['flow_id'].transform('size'))

# 2  Train/test split by flow, shared by all models of this dataset and seed
flow_ids   = df['flow_id'].unique()
flow_sizes = df.groupby('flow_id').size()
train_flows, test_flows, SPLIT_DRAW = create_or_load_split(Path(OUT_DIR) / 'split.json', flow_ids, flow_sizes,
                                                          TEST_SIZE, SEED, MIN_TRAIN_PKT_SHARE)
TRAIN_PKT_SHARE = float(flow_sizes.loc[train_flows].sum() / flow_sizes.sum())
print(f'split: {len(train_flows)} train / {len(test_flows)} test flows, the training flows hold '
      f'{TRAIN_PKT_SHARE:.1%} of the packets (draw {SPLIT_DRAW})')
val_flows = np.asarray([])
if VAL_FRACTION > 0:        # validation flows are taken from the training flows, with the same rule
    train_flows, val_flows, _, _ = constrained_split(train_flows, flow_sizes, VAL_FRACTION, SEED, MIN_TRAIN_PKT_SHARE)

# 3  Tail threshold tau and normalization, both from the training flows only
train_mask = df['flow_id'].isin(train_flows)
T_tail = np.quantile(np.log1p(df.loc[train_mask, 'time_diff'].to_numpy()), TAIL_Q)    # tau in log(1 + s)
use_idle_state = not ABLATION_NO_IDLE
print(f'tail threshold = {np.expm1(T_tail):.4f} s (q = {TAIL_Q}), idle state: {use_idle_state}')
scaler_len = StandardScaler().fit(df.loc[train_mask, ['flow_len_log']])
df['len_z'] = scaler_len.transform(df[['flow_len_log']])                               # xi
scaler_pkt = StandardScaler().fit(df.loc[train_mask, ['payload_log', 'time_diff_log']])
df[['plog_z', 'dt_z']] = scaler_pkt.transform(df[['payload_log', 'time_diff_log']])  # z

In [ ]:
# 4  HMM: k-means initialization of the core states, idle state at the tail threshold, EM
def build_hmm_matrix(frame):
    seqs, lens = [], []
    for _, grp in frame.groupby('flow_id'):
        arr = grp[['plog_z', 'dt_z']].to_numpy(dtype=np.float64)
        seqs.append(arr)
        lens.append(len(arr))
    return np.vstack(seqs), lens


X_train, len_train = build_hmm_matrix(df[df['flow_id'].isin(train_flows)])
K_STATES = K_CORE + (1 if use_idle_state else 0)
IDLE_K = K_STATES - 1 if use_idle_state else -1

trans_prior = np.full((K_STATES, K_STATES), LAMBDA_OFF)          # Dirichlet pseudo-counts Lambda
np.fill_diagonal(trans_prior, LAMBDA_SELF)
if use_idle_state:
    trans_prior[IDLE_K] = LAMBDA_LEAK
    trans_prior[IDLE_K, IDLE_K] = LAMBDA_IDLE
start_prior = np.full(K_STATES, 1.0 / K_STATES)
dt_tail_z = (T_tail - scaler_pkt.mean_[1]) / scaler_pkt.scale_[1]   # tau in the normalized space


def fit_hmm(init_seed):
    means = np.zeros((K_STATES, N_MIX_HMM, X_train.shape[1]))
    covars = np.zeros_like(means)
    km = KMeans(n_clusters=K_CORE, n_init=10, random_state=init_seed, algorithm='lloyd').fit(X_train)
    for k in range(K_CORE):
        means[k] = km.cluster_centers_[k]
        covars[k] = X_train[km.labels_ == k].var(axis=0) + INIT_VAR_EPS
    if use_idle_state:
        means[IDLE_K] = [np.median(X_train[:, 0]), dt_tail_z]
        covars[IDLE_K] = IDLE_INIT_VAR
    model = SafeGMMHMM(n_components=K_STATES, n_mix=N_MIX_HMM, covariance_type='diag', random_state=init_seed,
                       n_iter=MAX_ITER_HMM, min_covar=MIN_COVAR, init_params='', params='stmc')
    model.startprob_ = start_prior
    model.startprob_prior = start_prior
    model.transmat_ = np.ones_like(trans_prior) / K_STATES
    model.transmat_prior = trans_prior
    model.weights_ = np.full((K_STATES, N_MIX_HMM), 1.0 / N_MIX_HMM)
    model.means_ = means
    model.covars_ = covars
    return model.fit(X_train, len_train)


def hmm_is_valid(model):
    # EM can degenerate, e.g. when a state loses all posterior mass; the start, transition or
    # mixture weights then contain NaN and the posteriors cannot be computed
    arrays = (model.startprob_, model.transmat_, model.weights_)
    if not all(np.all(np.isfinite(x)) for x in arrays):
        return False
    return bool(np.isclose(model.startprob_.sum(), 1.0) and np.allclose(model.transmat_.sum(1), 1.0)
                and np.allclose(model.weights_.sum(1), 1.0))


for HMM_ATTEMPT in range(1, HMM_RESTARTS + 1):
    hmm_boot = fit_hmm(SEED + 1000 * (HMM_ATTEMPT - 1))
    if hmm_is_valid(hmm_boot):
        break
    print(f'the HMM fit degenerated (attempt {HMM_ATTEMPT} of {HMM_RESTARTS}): new initialization')
else:
    raise RuntimeError(f'the HMM fit degenerated in all {HMM_RESTARTS} attempts')
hmm_boot.__class__ = hmm.GMMHMM     # saved as a plain hmmlearn model
print(f'HMM log-likelihood of the training flows: {hmm_boot.score(X_train, len_train):.2f} (attempt {HMM_ATTEMPT})')
hmm_boot.covars_ = np.where(np.isfinite(hmm_boot.covars_), np.maximum(hmm_boot.covars_, MIN_COVAR), MIN_COVAR)
hmm_boot.means_[~np.isfinite(hmm_boot.means_)] = 0.0

# 5  State posteriors gamma of every packet (forward-backward on each flow)
gamma = np.vstack([hmm_boot.predict_proba(grp[['plog_z', 'dt_z']].to_numpy()) for _, grp in df.groupby('flow_id')])
df[[f'state_post_{k}' for k in range(K_STATES)]] = gamma

In [ ]:
# 6  MDN training data: one sample per (packet, state) with the posterior as weight
train_df = df[df['flow_id'].isin(train_flows)].reset_index(drop=True)
train_ds = PosteriorDataset(train_df, K_STATES, IDLE_K, use_flow_len=not ABLATION_NO_FLOW_LEN,
                            thresh_core=THRESH_GAMMA_CORE, thresh_idle=THRESH_GAMMA_IDLE)
train_dl = DataLoader(train_ds, batch_size=MDN_BATCHSIZE, shuffle=True, num_workers=4, pin_memory=True,
                      persistent_workers=True)

# 7  Training: weighted negative log-likelihood, Adam, gradient clipping, mixed precision on the GPU
model_mdn = GlobalStateLenMDN(K_STATES, hidden=HIDDEN_MDN, n_mix=N_MIX_MDN,
                              use_flow_len=not ABLATION_NO_FLOW_LEN).to(device)
optimiser = torch.optim.Adam(model_mdn.parameters(), lr=LR_MDN)
grad_scaler = torch.amp.GradScaler()
for epoch in range(1, EPOCHS_MDN + 1):
    model_mdn.train()
    total_loss = 0.0
    for x_feat, y_tgt, w in train_dl:
        x_feat, y_tgt, w = x_feat.to(device), y_tgt.to(device), w.to(device)
        optimiser.zero_grad()
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            pi, mu, sg, nu = model_mdn(x_feat)
            if DIST_FAMILY == 'student':
                nll = nll_student_t(y_tgt, pi, mu, sg, nu)
            else:
                nll = nll_gaussian(y_tgt, pi, mu, sg)
            loss = (w * nll).sum() / w.sum()
        grad_scaler.scale(loss).backward()
        grad_scaler.unscale_(optimiser)
        nn.utils.clip_grad_norm_(model_mdn.parameters(), GRAD_CLIP)
        grad_scaler.step(optimiser)
        grad_scaler.update()
        total_loss += loss.item()
    print(f'epoch {epoch:02d}: weighted NLL {total_loss / len(train_dl):.4f}')

# diagnostics: stationary distribution of the chain, and the median nu of each state at xi = 0
eigvals, eigvecs = np.linalg.eig(hmm_boot.transmat_.T)
stationary = np.real(eigvecs[:, eigvals.argmax()]).ravel()
stationary /= stationary.sum()
print('stationary distribution:', np.round(stationary, 3))
with torch.no_grad():
    if ABLATION_NO_FLOW_LEN:
        x_probe = torch.eye(K_STATES, dtype=torch.float32)
    else:
        x_probe = torch.zeros(K_STATES, K_STATES + 1, dtype=torch.float32)
        x_probe[:, :K_STATES] = torch.eye(K_STATES)
    _, _, _, nu = model_mdn(x_probe.to(device))
print('median nu per state:', np.round(torch.median(nu.cpu().squeeze(-1), -1).values.numpy(), 2))

# validation NLL, only used for the selection of M and H
VAL_NLL = None
if VAL_FRACTION > 0:
    val_ds = PosteriorDataset(df[df['flow_id'].isin(val_flows)].reset_index(drop=True), K_STATES, IDLE_K,
                              use_flow_len=not ABLATION_NO_FLOW_LEN, thresh_core=THRESH_GAMMA_CORE,
                              thresh_idle=THRESH_GAMMA_IDLE)
    model_mdn.eval()
    tot = wsum = 0.0
    with torch.no_grad():
        for xb, yb, wb in DataLoader(val_ds, batch_size=MDN_BATCHSIZE):
            xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)
            pi, mu, sg, nu = model_mdn(xb)
            nll = nll_student_t(yb, pi, mu, sg, nu) if DIST_FAMILY == 'student' else nll_gaussian(yb, pi, mu, sg)
            tot += float((wb * nll).sum())
            wsum += float(wb.sum())
    VAL_NLL = tot / wsum
    print('validation weighted NLL:', VAL_NLL)

In [ ]:
# 8  Generation: one synthetic flow per test flow
flow_len_for = flow_length_selector(FLOW_LEN_MODE, flow_sizes, train_flows, SEED)
synth_df = generate_flows(model_mdn, hmm_boot.transmat_, hmm_boot.startprob_, scaler_pkt, scaler_len,
                          test_flows, flow_len_for, idle_state=IDLE_K, temp_idle=TEMPERATURE_IDLE,
                          dist_family=DIST_FAMILY, use_flow_len=not ABLATION_NO_FLOW_LEN, seed=SEED,
                          device=device)
print(f'generated {len(synth_df):,} packets in {synth_df.flow_id.nunique()} flows')

In [ ]:
# 9  Outputs: synthetic flows, real test flows, trained model and metadata
out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)
atomic_write_csv(synth_df, out / f'{MODEL_KEY}.csv')
if not (out / 'test.csv').exists():
    atomic_write_csv(df.loc[df['flow_id'].isin(test_flows), ['flow_id', 'payload_length', 'time_diff']],
                     out / 'test.csv')
art = out / 'artifacts' / MODEL_KEY
art.mkdir(parents=True, exist_ok=True)
joblib.dump(hmm_boot, art / 'hmm.pkl')
joblib.dump(scaler_pkt, art / 'scaler_pkt.pkl')
joblib.dump(scaler_len, art / 'scaler_len.pkl')
torch.save(model_mdn.state_dict(), art / 'mdn.pth')
post_mass = df.loc[df['flow_id'].isin(train_flows), [f'state_post_{k}' for k in range(K_STATES)]].sum().to_numpy()
UNUSED_STATES = [int(k) for k in np.flatnonzero(post_mass < 1e-6)]   # never visited in generation
if UNUSED_STATES:
    print(f'HMM states {UNUSED_STATES} received no posterior mass')
meta = dict(dataset=DATASET, seed=SEED, K=int(K_STATES), K_core=K_CORE, idle=bool(use_idle_state),
            no_flow_len=ABLATION_NO_FLOW_LEN, dist_family=DIST_FAMILY, M=N_MIX_MDN, H=HIDDEN_MDN,
            hmm_n_mix=N_MIX_HMM, mdn_params=int(sum(p.numel() for p in model_mdn.parameters())),
            tail_q=TAIL_Q, tail_threshold_s=float(np.expm1(T_tail)), temperature_idle=TEMPERATURE_IDLE,
            flow_len_mode=FLOW_LEN_MODE, test_size=TEST_SIZE, val_fraction=VAL_FRACTION, val_nll=VAL_NLL,
            epochs=EPOCHS_MDN, n_train_flows=int(len(train_flows)), n_test_flows=int(len(test_flows)),
            hmm_attempts=int(HMM_ATTEMPT), split_draw=int(SPLIT_DRAW), train_packet_share=TRAIN_PKT_SHARE,
            unused_states=UNUSED_STATES)
(art / 'meta.json').write_text(json.dumps(meta, indent=2))